# Fase 1 — Etapas 2 y 3 Contrataciones Públicas DNCP Paraguay)
**Enfoque según `Anotaciones.md`**: unidad de análisis = **1 adjudicación** (`award id`).
Target = **¿aumentó el monto inicial adjudicado?** (`hubo_ampliacion_monto = 1` si existe enmienda *Ampliación de Monto* asociada al contrato de esa adjudicación en `con_amendments.csv`).
`contracts.csv` y `ten_tenderers.csv` se auditan pero se descartan del modelado (monto idéntico a adjudicación / solo 4 registros). `bid_details.csv` se descarta (columna única constante `valid`).


In [1]:
import os, unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "datos_completos"
OUT = "output"
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.max_columns", 50)
print("BASE:", BASE, "| archivos:", len([f for f in os.listdir(BASE) if f.endswith('.csv')]))


BASE: datos_completos | archivos: 54


## ETAPA 2 — Comprensión y selección de los datos
### 2.1 Inventario de archivos disponibles


In [2]:
import os, csv
# Inventario: tamano, filas (conteo rapido), columnas, periodo lo inferimos por columna fecha en 2.4/2.5
candidatos = ["awards.csv","con_amendments.csv","contracts.csv","awa_suppliers.csv",
              "bid_details.csv","ten_lots.csv","ten_items.csv","parties.csv","ten_tenderers.csv"]
inv = []
for f in candidatos:
    p = os.path.join(BASE, f)
    if not os.path.exists(p):
        inv.append([f, "NO EXISTE", None, None, None]); continue
    size_mb = round(os.path.getsize(p)/1024/1024, 2)
    with open(p, encoding="utf-8", errors="replace") as fh:
        r = csv.reader(fh)
        header = next(r)
        n = sum(1 for _ in r)
    inv.append([f, size_mb, n, len(header), "detalles en diccionario"])
inv_df = pd.DataFrame(inv, columns=["archivo","MB","filas","n_cols","periodo/nota"])
print(inv_df.to_string(index=False))
# Relaciones (diseno OCDS aplanado):
# releases(records) 1-N awards(por award id) 1-1 contracts(por awardID) 1-N amendments(por contract id)
# releases 1-N lots(por release id) | awards 1-1 suppliers(por award id) | tenderers/bids: cobertura minima
print("\nRelaciones clave:\n awards.award_id <-> contracts.awardID (1-1) <-> amendments.contract_id (1-N)\n awards.award_id <-> suppliers.award_id (1-1)\n release_id -> lots (1-N)")


           archivo    MB  filas  n_cols            periodo/nota
        awards.csv  2.10  14343       7 detalles en diccionario
con_amendments.csv  0.76   3735      17 detalles en diccionario
     contracts.csv  2.39  14339       9 detalles en diccionario
 awa_suppliers.csv  1.89  14343       4 detalles en diccionario
   bid_details.csv  0.19   2696       3 detalles en diccionario
      ten_lots.csv 13.86  71135      10 detalles en diccionario
     ten_items.csv 64.51 192087      16 detalles en diccionario
       parties.csv 20.86  68933      22 detalles en diccionario
 ten_tenderers.csv  0.00      4       4 detalles en diccionario

Relaciones clave:
 awards.award_id <-> contracts.awardID (1-1) <-> amendments.contract_id (1-N)
 awards.award_id <-> suppliers.award_id (1-1)
 release_id -> lots (1-N)


### 2.2 Diccionario de datos elaborado por el grupo
Se genera por código (no provisto por la cátedra). Tipos: observado vs. correcto. `%nulos` incluido.
Selección de columnas según `Anotaciones.md` marcada en `uso_previsto`.


In [3]:
def norm_txt(s):
    # normaliza encoding danado tipo 'Ampliaci�n' y minusculas/espacios
    if pd.isna(s): return s
    s = str(s).strip()
    s = s.replace("�","o")
    s = unicodedata.normalize("NFKD", s).encode("ascii","ignore").decode("ascii")
    return s

def diccionario(path, nrows=None, descripciones=None):
    df = pd.read_csv(path, nrows=nrows, encoding="utf-8", low_memory=False)
    rows = []
    for c in df.columns:
        s = df[c]
        miss = round(s.isna().mean()*100, 2)
        nun = s.nunique(dropna=True)
        if pd.api.types.is_numeric_dtype(s):
            rango = f"[{s.min()}, {s.max()}]"
        else:
            vals = s.dropna().unique()[:6].tolist()
            rango = f"{nun} unicos; ej: {vals}" if nun < 50 else f"{nun} unicos; ej: {vals[:3]}..."
        rows.append([os.path.basename(path), c, str(s.dtype), miss, rango,
                     (descripciones or {}).get(c, "")])
    return pd.DataFrame(rows, columns=["archivo","campo","tipo_obs","pct_nulos","rango/categorias","descripcion"]), df

desc_awards = {
  'compiledRelease/id': 'ID proceso/liberacion (agrupa adjudicaciones del mismo llamado)',
  'compiledRelease/awards/0/id': 'PK adjudicacion (award id)',
  'compiledRelease/awards/0/status': 'active/cancelled',
  'compiledRelease/awards/0/statusDetails': 'Adjudicado/Cancelado (redundante con status)',
  'compiledRelease/awards/0/date': 'Fecha adjudicacion (ISO8601 tz)',
  'compiledRelease/awards/0/value/amount': 'Monto inicial adjudicado (base del target)',
  'compiledRelease/awards/0/value/currency': 'Divisa (PYG 99.75%, USD 36 casos; se descarta tras auditoria)'}
d_aw, _ = diccionario(f"{BASE}/awards.csv", descripciones=desc_awards)
print(d_aw.to_string(index=False))
d_aw.to_csv(f"{OUT}/diccionario_awards.csv", index=False)


   archivo                                   campo tipo_obs  pct_nulos                                                                                                                                                                                  rango/categorias                                                     descripcion
awards.csv                      compiledRelease/id   object        0.0 8599 unicos; ej: ['204835-compra-medicamentos-1294743765', '206441-adquisicion-papel-p-uso-oficina-ad-ref-1295612345', '196208-adquisicion-desmalezadoras-motosierras-fumigadoras-1294412903']... ID proceso/liberacion (agrupa adjudicaciones del mismo llamado)
awards.csv             compiledRelease/awards/0/id   object        0.0                                                            14343 unicos; ej: ['204835-indufar-c-i-s-a-3', '204835-lourdes-elizabeth-peralta-rivas-2', '206441-gustavo-ramon-quevedo-ovelar-1']...                                      PK adjudicacion (award id)
awards.csv   

In [4]:
desc_amd = {'compiledRelease/id':'proceso','compiledRelease/contracts/0/id':'FK contrato (clave de cruce con contracts)',
  'compiledRelease/contracts/0/amendments/0/id':'PK enmienda','compiledRelease/contracts/0/amendments/0/date':'fecha enmienda',
  'compiledRelease/contracts/0/amendments/0/description':'TIPO enmienda: Ampliacion de Monto / Plazo / Renovacion Alquiler / Rescision (TARGET)',
  'compiledRelease/contracts/0/amendments/0/amendsAmount/amount':'monto ampliado (NaN si solo plazo; 0 nunca; ver anotacion: plazo=>monto cero/NaN)',
  'compiledRelease/contracts/0/amendments/0/amendsAmount/currency':'divisa enmienda',
  'compiledRelease/contracts/0/amendments/0/amendsPeriod/durationInDays':'dias extra si Ampliacion de Plazo'}
d_am, df_am_full = diccionario(f"{BASE}/con_amendments.csv", descripciones=desc_amd)
print(d_am.to_string(index=False))
print("\nDistribucion tipo enmienda:"); print(df_am_full['compiledRelease/contracts/0/amendments/0/description'].value_counts(dropna=False))
d_am.to_csv(f"{OUT}/diccionario_amendments.csv", index=False)


           archivo                                                                campo tipo_obs  pct_nulos                                                                                                                                                                                                                                    rango/categorias                                                                           descripcion
con_amendments.csv                                                   compiledRelease/id   object       0.00 1639 unicos; ej: ['190915-lp0583-adecuacion-lineas-media-tension-alimentadores-cae3-cae4-cae5-par4-1294218667', '206004-adquisicion-utiles-oficina-ad-referendum-1294674969', '206344-adquisicion-repuestos-vehiculos-ad-referendum-1295532971']...                                                                               proceso
con_amendments.csv                                       compiledRelease/contracts/0/id   object       0.00                 

In [5]:
# Resto de archivos: perfil resumido (completo en informe)
for f, desc in [
  ("contracts.csv", {'compiledRelease/contracts/0/id':'PK contrato','compiledRelease/contracts/0/awardID':'FK -> award id','compiledRelease/contracts/0/value/amount':'monto contrato (auditoria: identico a adjudicacion)'}),
  ("awa_suppliers.csv", {'compiledRelease/awards/0/id':'FK award','compiledRelease/awards/0/suppliers/0/id':'RUC proveedor (PY-RUC-...)','compiledRelease/awards/0/suppliers/0/name':'razon social'}),
  ("bid_details.csv", {}), ("ten_lots.csv", {}), ("ten_tenderers.csv", {}),
  ("ten_items.csv", {}), ("parties.csv", {})]:
    d, _ = diccionario(f"{BASE}/{f}", nrows=5000, descripciones=desc)
    print("="*100); print(f); print(d.to_string(index=False))


contracts.csv
      archivo                                             campo tipo_obs  pct_nulos                                                                                                                                                                                  rango/categorias                                         descripcion
contracts.csv                                compiledRelease/id   object        0.0 3118 unicos; ej: ['204835-compra-medicamentos-1294743765', '206441-adquisicion-papel-p-uso-oficina-ad-ref-1295612345', '196208-adquisicion-desmalezadoras-motosierras-fumigadoras-1294412903']...                                                    
contracts.csv                    compiledRelease/contracts/0/id   object        0.0                                                                                                               5000 unicos; ej: ['CO-22005-11-20195', 'CO-22005-11-20196', 'CD-30208-11-20266']...                                         PK contr

### 2.3 Unidad de análisis y clave primaria (verificación empírica de unicidad)
Unidad: **1 fila = 1 adjudicación**. PK: `awards/0/id`. FKs: `contracts.awardID`, `suppliers.award_id`, `amendments.contract_id` vía contrato.


In [6]:
awa = pd.read_csv(f"{BASE}/awards.csv")
con = pd.read_csv(f"{BASE}/contracts.csv")
sup = pd.read_csv(f"{BASE}/awa_suppliers.csv")
amd = pd.read_csv(f"{BASE}/con_amendments.csv")
lots = pd.read_csv(f"{BASE}/ten_lots.csv", usecols=["compiledRelease/id","compiledRelease/tender/lots/0/id"])
for name, df, key in [("awards",awa,"compiledRelease/awards/0/id"),("contracts(id)",con,"compiledRelease/contracts/0/id"),
                      ("contracts(awardID)",con,"compiledRelease/contracts/0/awardID"),("suppliers",sup,"compiledRelease/awards/0/id"),
                      ("amendments",amd,"compiledRelease/contracts/0/amendments/0/id")]:
    print(f"{name}: filas={len(df)} unicos={df[key].nunique()} duplicados_PK={df.duplicated(subset=[key]).sum()} -> {'PK OK' if df[key].nunique()==len(df) else 'NO es PK (relacion N:1 por diseno)'}")
print("lots: PK compuesta (release+lot):", lots.duplicated(subset=['compiledRelease/id','compiledRelease/tender/lots/0/id']).sum(), "dups sobre", len(lots))
print("awards por proceso (media):", round(awa.groupby('compiledRelease/id').size().mean(),3), "| contratos matchean awards:", con['compiledRelease/contracts/0/awardID'].isin(awa['compiledRelease/awards/0/id']).sum(), "/", len(con))


awards: filas=14343 unicos=14343 duplicados_PK=0 -> PK OK
contracts(id): filas=14339 unicos=14339 duplicados_PK=0 -> PK OK
contracts(awardID): filas=14339 unicos=14339 duplicados_PK=0 -> PK OK
suppliers: filas=14343 unicos=14343 duplicados_PK=0 -> PK OK
amendments: filas=3735 unicos=3735 duplicados_PK=0 -> PK OK
lots: PK compuesta (release+lot): 0 dups sobre 71135
awards por proceso (media): 1.668 | contratos matchean awards: 14339 / 14339


### 2.4 Perfilado inicial
Dimensiones, tipos detectados vs. correctos, cardinalidad categóricas, distribución preliminar numéricas.


In [ ]:
print("DIMENSIONES: awards", awa.shape, "| amendments", amd.shape, "| contracts", con.shape, "| suppliers", sup.shape, "| lots(71135,10) ten_items(192087,16) parties(69142,22) ten_tenderers(4,4) bid_details(2696,3)")
print("\nTIPOS: detectado object/float/int; CORRECTO: fechas->datetime, montos->float/int, ids/RUC->string categorica, status->category")
print("\nCARDINALIDAD awards:"); print(awa.nunique().to_string())
print("\nNUMERICAS awards.monto:"); print(awa['compiledRelease/awards/0/value/amount'].describe(percentiles=[.05,.25,.5,.75,.95]).to_string())
print("negativos:", (awa['compiledRelease/awards/0/value/amount']<0).sum(), "| ceros:", (awa['compiledRelease/awards/0/value/amount']==0).sum())
print("\nCATEGORICAS: awards.status:", awa['compiledRelease/awards/0/status'].value_counts(dropna=False).to_dict())
print("awards.divisa:", awa['compiledRelease/awards/0/value/currency'].value_counts(dropna=False).to_dict())
print("amendments.divisa nulos %:", round(amd['compiledRelease/contracts/0/amendments/0/amendsAmount/currency'].isna().mean()*100,2))
bid = pd.read_csv(f"{BASE}/bid_details.csv"); print("bid_details.status:", bid['compiledRelease/bids/details/0/status'].unique(), "(constante => sin poder predictivo)")


### 2.5 Evaluación de calidad
Completitud, consistencia, exactitud aparente, actualidad, trazabilidad.


In [ ]:
print("COMPLETITUD % nulos — awards:"); print((awa.isnull().mean()*100).round(2).to_string())
print("\namendments:"); print((amd.isnull().mean()*100).round(2).to_string())
print("\nCONSISTENCIA: monto contrato == monto adjudicacion en", (pd.merge(awa,con,left_on='compiledRelease/awards/0/id',right_on='compiledRelease/contracts/0/awardID')[['compiledRelease/awards/0/value/amount','compiledRelease/contracts/0/value/amount']].apply(lambda r: r[0]==r[1],axis=1).mean()*100).round(2), "% (14339/14339) => contracts NO aporta variacion (confirma Anotaciones 20:21).")
print("awards sin contrato:", len(awa)-pd.merge(awa,con,left_on='compiledRelease/awards/0/id',right_on='compiledRelease/contracts/0/awardID',how='inner').shape[0], "(4 adjudicaciones sin contrato: nunca podran tener enmienda => target 0, se conservan con contract_id nulo)")
print("EXACTITUD: montos>=0 OK (1 cero, 0 negativos); USD=36 casos (0.25%) moneda mixta.")
print("Plazo: solo Ampliacion de Plazo trae durationInDays (940/1056 con dato; 74.8% nulo global = esperado, no es missing real). Monto: NaN solo en Plazo/Rescision (28.5% = esperado).")
print("Encoding: 'Ampliaci�n' danado por latin1/utf8 => normalizar a 'Ampliacion de Monto/Plazo'.")
print("ACTUALIDAD: adjudicaciones 2010-06/2013-03; firmas contrato 2002-2012 (hay firmas 2002 anteriores a adjudicacion: inconsistencia de carga, se audita); enmiendas 2010-11/2019-01 (ventana posterior OK para target).")
print("TRAZABILIDAD: OCDS DNCP https://www.contrataciones.gov.py/datos/data ; corte datos_completos; licencia acceso publico Ley 5282/2014 (uso academico, patrones agregados).")


### 2.6 Selección justificada (qué se conserva / descarta y por qué)
Cada descarte vinculado a objetivo. Respeta `Anotaciones.md`.


In [ ]:
sel = pd.DataFrame([
  ["awards: id proceso, award id, status, statusDetails, fecha, monto", "CONSERVAR (base + monto inicial + fecha orden temporal)", "Obj: target y proveedor-historial"],
  ["awards: divisa", "DESCARTAR tras auditoria (99.75% PYG; 36 USD se filtran)", "Anotaciones: todas menos divisa,id"],
  ["con_amendments: contract id, amendment id, fecha, descripcion, amendsAmount, durationInDays", "CONSERVAR (TARGET)", "Anotaciones: id, descripcion, monto, fecha (+duration si plazo)"],
  ["con_amendments: rationale/guarantee*/prepaid/financialCode", "DESCARTAR (>99.8% nulos o tramite, sin hipotesis)", "Obj: evitar ruido"],
  ["contracts: solo contract id + awardID (puente) + fecha firma + plazo", "CONSERVAR SOLO PUENTE; montos/status duplican awards", "Anotaciones+auditoria 100% iguales; no duplicar"],
  ["awa_suppliers: award id, RUC (+nombre validacion)", "CONSERVAR (id,ruc lista definitiva)", "Anotaciones: agrupar por proveedor, previas, frecuencia, patron temporal"],
  ["ten_lots: agregados por proceso (n_lotes, monto estimado, estado)", "CONSERVAR AGREGADO", "Anotaciones: cantidad/tipo/valor/estado lote"],
  ["parties: RUC/nombre/rol", "CONSERVAR SOLO VALIDACION RUC; resto contacto/direccion/tel DESCARTAR", "Anotaciones parties"],
  ["bid_details (2696, status=valid cte)", "DESCARTAR", "Anotaciones: ninguna util"],
  ["ten_tenderers (4 filas)", "DESCARTAR (auditar)", "Anotaciones 20:21: 4 registros no utiles"],
  ["ten_items (192087 filas, granularidad item)", "DESCARTAR del modelo (peso/cobertura parcial, sin FK award)", "Alcance: lote ya captura escala"],
  ["records/tender 1: metodo_contratacion + detalles_metodo_contratacion", "CONSERVAR (open/direct; Contratacion Directa vs LPN/CO)", "Hipotesis: menor competencia -> mas ampliacion"],
  ["records/tender 2: categoria_principal_contratacion (goods/services/works)", "CONSERVAR + FILTRO FINAL (solo goods/services/works)", "Alcance pedido: obras/compras/servicios; works = mas ampliaciones"],
  ["records/tender 3: monto_licitacion + monto_presupuesto", "CONSERVAR + ratios vs monto_adj", "Hipotesis: mala estimacion -> ampliacion"],
  ["records/tender 4: criterio_adjudicacion + detalles_criterio (Por Item/Total/Lote)", "CONSERVAR", "Estructura del llamado; cruza con n_lotes"],
  ["records/tender 5: duracion_periodo_licitacion/consultas + velocidad_proceso", "CONSERVAR (plazos cortos = apuro)", "Hipotesis: plazos cortos -> mas enmiendas"],
  ["records/tender 6: fechas publicacion/inicio/fin licitacion + inicio adjudicacion", "CONSERVAR + deltas temporales", "Estacionalidad y friccion del proceso"],
  ["records: cantidad_oferentes / subasta_electronica / direcciones / moneda_tender", "DESCARTAR (100%/94.5% nulos o texto libre)", "Sin cobertura para modelar"],
], columns=["decision","detalle","vinculo"])
print(sel.to_string(index=False))


### 2.7 Documentación de la fuente


In [ ]:
import datetime
print("URL exacta: https://www.contrataciones.gov.py/datos/data (estandar OCDS)")
print("Corte local: datos_completos/ (awards 14343, contracts 14339, amendments 3735, suppliers 14343, lots 71135, items 192087, parties 69142)")
print("Fecha de descarga/copia local: registrar aqui por el grupo (hoy:", datetime.date.today(), ")")
print("Version: archivos .sha incluidos por tabla; licencia: datos abiertos Ley 5282/2014, uso academico; unidades: montos PYG (USD minoritario filtrado), fechas ISO8601, plazos en dias.")


## ETAPA 3 — Limpieza y transformación
Regla no negociable: todo reproducible por código desde archivos originales (este notebook).


In [ ]:
bitacora = []
def log(paso, afectados, fundamento):
    bitacora.append([paso, afectados, fundamento]); print(f"[BITACORA] {paso} | afectados={afectados} | {fundamento}")


### 3.1 Valores faltantes (cuantificación, mecanismo, estrategia)


In [ ]:
# Awards: 0% nulos -> sin accion. Amendments: nulos ESTRUCTURALES (no MCAR): monto NaN <=> Plazo/Rescision; dias NaN <=> no-Plazo.
print((amd.isnull().mean()*100).round(2).to_string())
log("3.1 conservar NaN estructurales (monto en Plazo/Rescision; dias en no-Plazo)", int(amd['compiledRelease/contracts/0/amendments/0/amendsAmount/amount'].isna().sum()), "Ausencia por diseno OCDS, no imputable; el TARGET usa descripcion normalizada, no el monto")
log("3.1 descartar columnas guarantee*/rationale/prepaid (>99.8% nulas)", 7, "Sin hipotesis asociada; imputar inventaria datos")


### 3.2 Duplicados exactos y parciales


In [ ]:
print("exactos:", {"awards":awa.duplicated().sum(),"contracts":con.duplicated().sum(),"suppliers":sup.duplicated().sum(),"amendments":amd.duplicated().sum()})
print("PK parcial awards:", awa.duplicated(subset=['compiledRelease/awards/0/id']).sum(), "| suppliers:", sup.duplicated(subset=['compiledRelease/awards/0/id']).sum())
log("3.2 sin eliminacion (0 duplicados exactos y 0 PK parcial en base)", 0, "Relaciones N:1 (lots/items) son por diseno, se agregan no se deducen")


### 3.3 Atípicos e inconsistencias


In [ ]:
# Montos: 1 cero, max 1.45e12 (outlier extremo, se conserva + log); fechas firma 2002 previas a adjudicacion 2010 (error carga, se conserva pero no se usa como feature dura)
print("monto max:", awa['compiledRelease/awards/0/value/amount'].max(), "| p99:", awa['compiledRelease/awards/0/value/amount'].quantile(.99))
print("encoding descripcion unica (cruda):", amd['compiledRelease/contracts/0/amendments/0/description'].unique())
amd['tipo_norm'] = amd['compiledRelease/contracts/0/amendments/0/description'].apply(norm_txt)
print("normalizada:", amd['tipo_norm'].unique())
log("3.3 normalizar encoding/lower-strip (Ampliacion de Monto/Plazo, Renovacion, Rescision)", len(amd), "Misma semantica escrita distinta por codificacion")
log("3.3 NO winsorizar montos (conservar + usar log_monto)", 1, "Outliers son senal (contratos grandes); recortar sesgaria target")
log("3.3 filtrar 36 USD + 39 cancelled (separado)", 75, "Moneda no comparable / canceladas no pueden ampliarse; se reportan, no se modelan")


### 3.4 Normalización de tipos y formatos


In [ ]:
awa['fecha_adj'] = pd.to_datetime(awa['compiledRelease/awards/0/date'], errors='coerce', utc=True).dt.tz_convert(None)
con['fecha_firma'] = pd.to_datetime(con['compiledRelease/contracts/0/dateSigned'], errors='coerce', utc=True).dt.tz_convert(None)
amd['fecha_enm'] = pd.to_datetime(amd['compiledRelease/contracts/0/amendments/0/date'], errors='coerce', utc=True).dt.tz_convert(None)
for c in ['compiledRelease/awards/0/status','compiledRelease/awards/0/statusDetails','compiledRelease/awards/0/value/currency']:
    awa[c] = awa[c].astype(str).str.strip()
sup['compiledRelease/awards/0/suppliers/0/id'] = sup['compiledRelease/awards/0/suppliers/0/id'].astype(str).str.strip().str.upper()
awa['compiledRelease/awards/0/value/amount'] = pd.to_numeric(awa['compiledRelease/awards/0/value/amount'], errors='coerce')
print("nulos fecha_adj:", awa['fecha_adj'].isna().sum(), "| fecha_enm:", amd['fecha_enm'].isna().sum())
log("3.4 fechas a datetime tz-naive, RUC upper-strip, montos a numerico", len(awa), "Comparabilidad y orden temporal para historial proveedor sin leakage")


### 3.5 Variables derivadas (con corte justificado y sin leakage)
Proveedor-historial usa **solo adjudicaciones estrictamente anteriores** (por fecha). `dias_hasta_enmienda` es descriptivo, NO feature predictiva.


In [ ]:
AW, AID, MON, REL = 'compiledRelease/awards/0/id','compiledRelease/contracts/0/awardID','compiledRelease/contracts/0/id','compiledRelease/id'
# 1) Agregado enmiendas por contrato + flag target
amd['es_monto'] = amd['tipo_norm'].str.contains('Monto', na=False)
amd['es_plazo'] = amd['tipo_norm'].str.contains('Plazo', na=False)
g = amd.groupby('compiledRelease/contracts/0/id').agg(
    n_enmiendas=('compiledRelease/contracts/0/amendments/0/id','count'),
    hubo_ampl_monto=('es_monto','max'),
    hubo_ampl_plazo=('es_plazo','max'),
    monto_ampliado_total=('compiledRelease/contracts/0/amendments/0/amendsAmount/amount','sum'),
    dias_ampliacion_total=('compiledRelease/contracts/0/amendments/0/amendsPeriod/durationInDays','sum'),
    fecha_primera_enm=('fecha_enm','min'))
g['hubo_ampl_monto'] = g['hubo_ampl_monto'].fillna(False).astype(int)
g['hubo_ampl_plazo'] = g['hubo_ampl_plazo'].fillna(False).astype(int)
print("contratos con ampl.monto:", g['hubo_ampl_monto'].sum(), "/", len(con), "| tasa adjudicacion ~10.7%")
# 2) Lotes por proceso
lots_full = pd.read_csv(f"{BASE}/ten_lots.csv")
lag = lots_full.groupby('compiledRelease/id').agg(n_lotes=('compiledRelease/tender/lots/0/id','count'),
    monto_estimado_lotes=('compiledRelease/tender/lots/0/value/amount','sum'))
print("lotes agregados:", lag.shape)
log("3.5 target=hubo_ampliacion_monto (Ampliacion de Monto; Renovacion excluida por ser alquiler)", int(g['hubo_ampl_monto'].sum()), "Definicion estricta de 'aumento del monto inicial'; tasa 10.7% apta para clasificacion")


In [ ]:
# 2b) Tender/planning desde records.csv (variables 1-6, join por release_id; todo conocido ANTES de adjudicar)
usos = ['compiledRelease/id',
 'compiledRelease/tender/procurementMethod','compiledRelease/tender/procurementMethodDetails',
 'compiledRelease/tender/mainProcurementCategory',
 'compiledRelease/tender/value/amount',
 'compiledRelease/planning/budget/amount/amount',
 'compiledRelease/tender/awardCriteria','compiledRelease/tender/awardCriteriaDetails',
 'compiledRelease/tender/tenderPeriod/durationInDays','compiledRelease/tender/enquiryPeriod/durationInDays',
 'compiledRelease/tender/datePublished','compiledRelease/tender/tenderPeriod/startDate',
 'compiledRelease/tender/tenderPeriod/endDate','compiledRelease/tender/awardPeriod/startDate']
rec = pd.read_csv(f"{BASE}/records.csv", usecols=usos, low_memory=False)
print("records:", rec.shape, "| releases unicos:", rec['compiledRelease/id'].nunique(), "| dups:", rec.duplicated(subset=['compiledRelease/id']).sum())
ten = rec.rename(columns={'compiledRelease/id':'release_id',
 'compiledRelease/tender/procurementMethod':'metodo_contratacion',
 'compiledRelease/tender/procurementMethodDetails':'det_metodo_contratacion',
 'compiledRelease/tender/mainProcurementCategory':'categoria_principal',
 'compiledRelease/tender/value/amount':'monto_licitacion',
 'compiledRelease/planning/budget/amount/amount':'monto_presupuesto',
 'compiledRelease/tender/awardCriteria':'criterio_adj',
 'compiledRelease/tender/awardCriteriaDetails':'det_criterio_adj',
 'compiledRelease/tender/tenderPeriod/durationInDays':'dur_licitacion_dias',
 'compiledRelease/tender/enquiryPeriod/durationInDays':'dur_consultas_dias',
 'compiledRelease/tender/datePublished':'fecha_pub_lic',
 'compiledRelease/tender/tenderPeriod/startDate':'fecha_ini_lic',
 'compiledRelease/tender/tenderPeriod/endDate':'fecha_fin_lic',
 'compiledRelease/tender/awardPeriod/startDate':'fecha_ini_adj_periodo'})
# normalizacion texto (encoding danado: Contrataci�n/Licitaci�n) y fechas
for c in ['metodo_contratacion','det_metodo_contratacion','categoria_principal','criterio_adj','det_criterio_adj']:
    ten[c] = ten[c].apply(norm_txt)
for c in ['fecha_pub_lic','fecha_ini_lic','fecha_fin_lic','fecha_ini_adj_periodo']:
    ten[c] = pd.to_datetime(ten[c], errors='coerce', utc=True).dt.tz_convert(None)
ten['velocidad_proceso_dias'] = ten['dur_licitacion_dias'].fillna(0) + ten['dur_consultas_dias'].fillna(0)
print("metodo:", ten['metodo_contratacion'].value_counts(dropna=False).to_dict())
print("categoria:", ten['categoria_principal'].value_counts(dropna=False).to_dict())
print("criterio:", ten['det_criterio_adj'].value_counts(dropna=False).to_dict())
log("3.5 tender join: 6 bloques desde records (metodo/categoria/montos/criterio/duraciones/fechas)", len(ten), "Todo pre-adjudicacion, sin leakage; match 100% awards en auditoria 3.6")

In [ ]:
# 3) Base + puente contrato + proveedor + historial temporal sin leakage
base = awa.rename(columns={'compiledRelease/awards/0/id':'award_id','compiledRelease/id':'release_id',
  'compiledRelease/awards/0/status':'estado','compiledRelease/awards/0/statusDetails':'estado_det',
  'compiledRelease/awards/0/value/amount':'monto_adj','compiledRelease/awards/0/value/currency':'moneda'})[
  ['award_id','release_id','estado','estado_det','moneda','monto_adj','fecha_adj']]
puente = con.rename(columns={'compiledRelease/contracts/0/awardID':'award_id','compiledRelease/contracts/0/id':'contract_id',
  'compiledRelease/contracts/0/period/durationInDays':'plazo_dias'})[['award_id','contract_id','fecha_firma','plazo_dias']]
df = base.merge(puente, on='award_id', how='left')
df = df.merge(g, left_on='contract_id', right_index=True, how='left')
for c in ['n_enmiendas','hubo_ampl_monto','hubo_ampl_plazo','monto_ampliado_total','dias_ampliacion_total']:
    df[c] = df[c].fillna(0)
df['dias_hasta_enmienda'] = (df['fecha_primera_enm'] - df['fecha_adj']).dt.days  # solo descriptivo
df = df.merge(lag, left_on='release_id', right_index=True, how='left')
df = df.merge(ten, on='release_id', how='left')
# derivados tender (necesitan monto_adj): ratios y deltas temporales
df['ratio_adj_vs_licitacion'] = df['monto_adj'] / df['monto_licitacion']
df['brecha_licitacion_vs_presupuesto'] = df['monto_licitacion'] / df['monto_presupuesto']
df['dias_pub_a_adj'] = (df['fecha_adj'] - df['fecha_pub_lic']).dt.days
df['dias_lic_a_adj'] = (df['fecha_adj'] - df['fecha_fin_lic']).dt.days
s = sup.rename(columns={'compiledRelease/awards/0/id':'award_id','compiledRelease/awards/0/suppliers/0/id':'ruc','compiledRelease/awards/0/suppliers/0/name':'proveedor'})[['award_id','ruc','proveedor']]
df = df.merge(s, on='award_id', how='left')
# historial proveedor: ordenar por fecha y contar solo previas
df = df.sort_values(['ruc','fecha_adj'])
df['prov_n_previas'] = df.groupby('ruc').cumcount()
df['prov_monto_previo_acum'] = df.groupby('ruc')['monto_adj'].transform(lambda x: x.cumsum().shift(1).fillna(0))
first = df.groupby('ruc')['fecha_adj'].transform('min')
df['prov_antiguedad_dias'] = (df['fecha_adj'] - first).dt.days
df['anio'] = df['fecha_adj'].dt.year; df['mes'] = df['fecha_adj'].dt.month
df['log_monto'] = np.log1p(df['monto_adj'])
df['ratio_ampliacion'] = np.where(df['hubo_ampl_monto']==1, df['monto_ampliado_total']/df['monto_adj'], 0)
print(df.shape); print(df[['hubo_ampl_monto']].mean().to_string())
log("3.5 historial proveedor sin leakage (cumcount/cumsum shift por fecha)", len(df), "Evita usar futuro para predecir pasado; patron temporal pedido en Anotaciones")


### 3.6 Integración de fuentes (clave, join, auditoría)


In [ ]:
print(f"awards base: {len(base)} | tras LEFT contrato: {len(df)} (anti-duplicacion: {df['award_id'].nunique()==len(df)})")
print("no apareados: sin contrato:", df['contract_id'].isna().sum(), "| sin lotes:", df['n_lotes'].isna().sum(), "| sin proveedor:", df['ruc'].isna().sum())
print("tender: sin match:", df['metodo_contratacion'].isna().sum(), "| categoria nula:", df['categoria_principal'].isna().sum(), "| monto_licit nulo:", df['monto_licitacion'].isna().sum(), "| monto_presup nulo:", df['monto_presupuesto'].isna().sum())
print("tender cobertura awards: metodo:", df['metodo_contratacion'].value_counts(dropna=False).to_dict())
print("categoria award-level:", df['categoria_principal'].value_counts(dropna=False).to_dict())
print("tipo join: LEFT desde awards (conservar 100% unidad analisis); granularidad: contrato/enmienda agregada a 1 fila/award (sin explosion); lots agregada por release.")
log("3.6 LEFT awards->contratos/enmiendas/lotes/proveedor (agregados, sin duplicar filas)", len(df), "Unidad 1 fila=1 adjudicacion preservada")


### 3.7 Bitácora + dataset limpio (reproducible)
Filtro final: solo `active/Adjudicado` + `PYG` (canceladas/USD auditadas aparte). Renovación de Alquiler NO cuenta como target.


In [ ]:
antes = len(df)
df_final = df[(df['estado']=='active') & (df['moneda']=='PYG')].copy()
log("3.7 filtro active+PYG", antes-len(df_final), "39 cancelled + 36 USD excluidos (no ampliables / no comparables)")
antes2 = len(df_final)
df_final = df_final[df_final['categoria_principal'].isin(['goods', 'services', 'works'])].copy()
log("3.7 filtro categoria goods/services/works (obras+compras+servicios)", antes2-len(df_final), "Alcance pedido; resto/indefinido fuera")
df_final['n_lotes'] = df_final['n_lotes'].fillna(0)
df_final['monto_estimado_lotes'] = df_final['monto_estimado_lotes'].fillna(0)
cols = ['award_id','release_id','fecha_adj','monto_adj','log_monto','anio','mes','contract_id','fecha_firma','plazo_dias',
        'ruc','proveedor','prov_n_previas','prov_monto_previo_acum','prov_antiguedad_dias',
        'n_lotes','monto_estimado_lotes',
        'metodo_contratacion','det_metodo_contratacion','categoria_principal',
        'monto_licitacion','monto_presupuesto','ratio_adj_vs_licitacion','brecha_licitacion_vs_presupuesto',
        'criterio_adj','det_criterio_adj','dur_licitacion_dias','dur_consultas_dias','velocidad_proceso_dias',
        'fecha_pub_lic','fecha_ini_lic','fecha_fin_lic','dias_pub_a_adj','dias_lic_a_adj',
        'n_enmiendas','hubo_ampl_plazo','monto_ampliado_total','dias_ampliacion_total',
        'dias_hasta_enmienda','ratio_ampliacion','hubo_ampl_monto']
df_final = df_final[cols].rename(columns={'hubo_ampl_monto':'target_ampliacion_monto'})
df_final.to_csv(f"{OUT}/dataset_final.csv", index=False)
pd.DataFrame(bitacora, columns=["decision","afectados","fundamento"]).to_csv(f"{OUT}/bitacora_limpieza.csv", index=False)
print("FINAL:", df_final.shape, "| target rate:", round(df_final['target_ampliacion_monto'].mean(),4), "| n positivos:", int(df_final['target_ampliacion_monto'].sum()))
print("guardado:", f"{OUT}/dataset_final.csv", "|", f"{OUT}/bitacora_limpieza.csv")
print(pd.DataFrame(bitacora, columns=["decision","afectados","fundamento"]).to_string(index=False))


In [ ]:
# Olivazo rapido del dataset final + duplicados finales
print("dups award_id final:", df_final.duplicated(subset=['award_id']).sum())
print(df_final.head(3).to_string())
print(df_final.describe(include='all').to_string()[:3000])
